# Day 1 — Spotify Streaming History Exploration

Goal: load my Spotify export, sanity-check the data, and produce the cleaned Parquet files that day 2+ will consume.

**Outputs we want by end of notebook:**
- `data/processed/all_streams.parquet` — every stream event
- `data/processed/real_plays.parquet` — plays >= 30s only
- `data/processed/plays_by_artist.parquet` — recommender input (artist-level)
- `data/processed/plays_by_track.parquet` — for analytics
- `data/processed/daily_listening.parquet` — for the dashboard

**Sanity questions to answer before moving on:**
1. How many plays total? Date range?
2. What's my top-N artists list? Does it match my intuition?
3. Skip rate — what fraction of streams are < 30s?
4. Any weird data quality issues (duplicates, missing fields, podcasts mixed in)?

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make the src package importable from the notebook
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import plotly.express as px
from spotify_recs import loader

## 1. Load raw streams

In [2]:
streams = loader.load_streams('../data/raw')
print(f'Total events: {len(streams):,}')
print(f'Date range:   {streams["ts"].min()} → {streams["ts"].max()}')
print(f'Span:         {(streams["ts"].max() - streams["ts"].min()).days} days')
streams.head()

  loaded StreamingHistory_music_0.json (account_data): 10,000 rows
  loaded StreamingHistory_music_1.json (account_data): 10,000 rows
  loaded StreamingHistory_music_10.json (account_data): 3,142 rows
  loaded StreamingHistory_music_2.json (account_data): 10,000 rows
  loaded StreamingHistory_music_3.json (account_data): 10,000 rows
  loaded StreamingHistory_music_4.json (account_data): 10,000 rows
  loaded StreamingHistory_music_5.json (account_data): 10,000 rows
  loaded StreamingHistory_music_6.json (account_data): 10,000 rows
  loaded StreamingHistory_music_7.json (account_data): 10,000 rows
  loaded StreamingHistory_music_8.json (account_data): 10,000 rows
  loaded StreamingHistory_music_9.json (account_data): 10,000 rows
Total events: 103,142
Date range:   2024-11-16 00:04:00+00:00 → 2025-11-16 21:17:00+00:00
Span:         365 days


,ts,track_name,artist_name,album_name,ms_played,track_uri,platform,country,shuffle,skipped,source
0,2024-11-16 00:04:00+00:00,You Ain't Gotta Lie (Momma Said),Kendrick Lamar,<NA>,18459,<NA>,<NA>,<NA>,None,None,account_data
1,2024-11-16 00:04:00+00:00,Not Enough,Little Brother,<NA>,1973,<NA>,<NA>,<NA>,None,None,account_data
2,2024-11-16 00:04:00+00:00,Pipe Down,Drake,<NA>,673,<NA>,<NA>,<NA>,None,None,account_data
3,2024-11-16 00:04:00+00:00,"Say You Love Me, One More Time",D.J. Rogers,<NA>,650,<NA>,<NA>,<NA>,None,None,account_data
4,2024-11-16 00:04:00+00:00,"God Is Fair, Sexy Nasty (feat. Kendrick Lamar)",Mac Miller,<NA>,1602,<NA>,<NA>,<NA>,None,None,account_data


In [3]:
# Field coverage — what fraction of rows have each optional field?
# (Tells us which export format(s) we have)
streams.notna().mean().sort_values(ascending=False).round(3)

ts             1.0
track_name     1.0
artist_name    1.0
ms_played      1.0
source         1.0
album_name     0.0
track_uri      0.0
platform       0.0
country        0.0
shuffle        0.0
skipped        0.0
dtype: float64

## 2. Skip rate & play threshold

In [4]:
# Distribution of play durations. The 30s threshold should be a natural cutoff.
fig = px.histogram(
    streams[streams['ms_played'] < 300_000],  # zoom to first 5 min
    x='ms_played',
    nbins=60,
    title='Distribution of play durations (capped at 5 min)',
)
fig.add_vline(x=30_000, line_dash='dash', annotation_text='30s threshold')
fig.show()

In [5]:
skip_rate = (streams['ms_played'] < 30_000).mean()
print(f'Skip rate (< 30s): {skip_rate:.1%}')
print(f'Plays kept: {(streams["ms_played"] >= 30_000).sum():,} / {len(streams):,}')

Skip rate (< 30s): 72.1%
Plays kept: 28,752 / 103,142


## 3. Top artists — gut check

In [6]:
plays = loader.filter_real_plays(streams)
by_artist = loader.aggregate_by_artist(plays)
by_artist.head(20)

,user_id,artist_name,play_count,total_minutes,first_played,last_played
0,me,Kanye West,607,2341.7,2024-11-16 00:33:00+00:00,2025-11-16 14:29:00+00:00
1,me,Malcolm Todd,485,1362.0,2024-11-27 23:49:00+00:00,2025-11-15 22:18:00+00:00
2,me,HYUKOH,312,1426.9,2025-01-07 16:38:00+00:00,2025-10-27 02:29:00+00:00
3,me,Underworld,308,2098.0,2024-11-16 22:28:00+00:00,2025-11-05 00:18:00+00:00
4,me,Daft Punk,294,1293.8,2024-11-20 04:06:00+00:00,2025-11-16 16:19:00+00:00
5,me,Kendrick Lamar,291,1194.6,2024-11-21 17:36:00+00:00,2025-11-16 20:52:00+00:00
6,me,D'Angelo,288,1309.8,2024-11-18 00:00:00+00:00,2025-11-15 15:02:00+00:00
7,me,JPEGMAFIA,279,770.3,2024-11-16 13:53:00+00:00,2025-11-11 20:08:00+00:00
8,me,Joey Valence & Brae,253,700.9,2024-12-01 15:19:00+00:00,2025-11-16 21:07:00+00:00
9,me,Freddie Gibbs,251,786.2,2024-11-18 17:22:00+00:00,2025-11-16 15:57:00+00:00


In [7]:
# Quick look at the long tail — most artists in your library are heard once or twice
fig = px.histogram(
    by_artist, x='play_count', log_y=True,
    title=f'Play count distribution per artist (n={len(by_artist):,} artists)',
    nbins=50,
)
fig.show()

print(f'Artists played only once: {(by_artist["play_count"] == 1).sum():,}')
print(f'Artists played 10+ times: {(by_artist["play_count"] >= 10).sum():,}')

Artists played only once: 1,626
Artists played 10+ times: 533


## 4. Listening over time

In [8]:
daily = loader.daily_listening(plays)
fig = px.line(daily, x='date', y='minutes', title='Daily listening (minutes)')
fig.update_traces(line_width=1)
fig.show()

In [9]:
# Hour-of-day pattern — when are you listening?
plays_local = plays.copy()
plays_local['hour'] = plays_local['ts'].dt.tz_convert('America/New_York').dt.hour
hourly = plays_local.groupby('hour').size().reset_index(name='plays')
fig = px.bar(hourly, x='hour', y='plays', title='Plays by hour of day (local time)')
fig.show()

## 5. Data quality checks

In [10]:
# Are there exact-duplicate plays? (same artist+track+timestamp)
dupes = streams.duplicated(subset=['ts', 'artist_name', 'track_name']).sum()
print(f'Exact duplicate rows: {dupes:,}')

# Any artists that look like podcasts/ambient junk that snuck through?
# (Long tail of single plays often catches these)
print('\nArtists with exactly 1 play (sample):')
by_artist[by_artist['play_count'] == 1]['artist_name'].sample(
    min(15, (by_artist['play_count'] == 1).sum()), random_state=0
).tolist()

Exact duplicate rows: 3,847

Artists with exactly 1 play (sample):


['seny',
 'coldstream',
 'Vangelis',
 'ARAGON',
 'Brownstone',
 'RC & The Gritz',
 'Foggieraw',
 'Trevor Dandy',
 'Fetty Wap',
 'mcgwn',
 'prod. DTM',
 'Sigma',
 'Swaine Delgado',
 'KMD',
 'Duke Hugh']

## 6. Save processed files

Once the above looks reasonable, write everything out as Parquet.

In [11]:
paths = loader.save_processed(streams, out_dir='../data/processed')
paths

  wrote all_streams.parquet: 103,142 rows
  wrote real_plays.parquet: 28,752 rows
  wrote plays_by_artist.parquet: 3,678 rows
  wrote plays_by_track.parquet: 7,648 rows
  wrote daily_listening.parquet: 353 rows


{'all_streams': PosixPath('../data/processed/all_streams.parquet'),
 'real_plays': PosixPath('../data/processed/real_plays.parquet'),
 'by_artist': PosixPath('../data/processed/plays_by_artist.parquet'),
 'by_track': PosixPath('../data/processed/plays_by_track.parquet'),
 'daily': PosixPath('../data/processed/daily_listening.parquet')}

---

## Day 1 checklist

- [ ] All JSON files loaded without errors
- [ ] Date range matches expectation (~1 year for Account Data)
- [ ] Top artists list passes the gut check
- [ ] Skip rate looks reasonable (typically 20–40%)
- [ ] No obvious data quality issues
- [ ] Parquet files written to `data/processed/`

Day 2 picks up from `plays_by_artist.parquet`.